### Data Ingestion


In [2]:
### document structure
from langchain_core.documents import Document
doc = Document(
    page_content="This is a main content i am using to create a RAG",
    metadata={"source": "exapmle.com", "author": "John Doe", "date": "2024-06-01"
    }
)
doc



Document(metadata={'source': 'exapmle.com', 'author': 'John Doe', 'date': '2024-06-01'}, page_content='This is a main content i am using to create a RAG')

In [ ]:
## create a simple txt file

import os
os.makedirs("../data/text_files", exist_ok=True)

In [ ]:
sample_text={
    "../data/text_files/python_intro.txt": """Machine Learnign basics:
    Machine learning is a subset of artificial intelligence where algorithms analyze data to learn patterns and make predictions or decisions without
    being explicitly programmed for every task.  Unlike traditional programming, which relies on fixed rules, machine learning models improve their accuracy over time as they process more experience (data). 

    Supervised Learning: Models are trained on labeled data (data with known answers) to predict outcomes for new inputs, commonly used for classification and regression tasks like spam detection or price forecasting. 
    Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden patterns, structures, or groupings, such as clustering similar customer behaviors or reducing data dimensions. 
    Reinforcement Learning: Agents learn to make sequences of decisions by receiving rewards or penalties for their actions, aiming to maximize cumulative reward in dynamic environments like gaming or robotics.
       
    """
}

for filepath, content in sample_text.items():
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content)

print("Sample text files created successfully.")

In [4]:
from langchain_community.document_loaders import TextLoader

loader = TextLoader("../data/text_files/python_intro.txt", encoding="utf-8")
docs = loader.load()
print(docs)




[Document(metadata={'source': '../data/text_files/python_intro.txt'}, page_content='Machine Learnign basics:\n    Machine learning is a subset of artificial intelligence where algorithms analyze data to learn patterns and make predictions or decisions without\n    being explicitly programmed for every task.  Unlike traditional programming, which relies on fixed rules, machine learning models improve their accuracy over time as they process more experience (data). \n\n    Supervised Learning: Models are trained on labeled data (data with known answers) to predict outcomes for new inputs, commonly used for classification and regression tasks like spam detection or price forecasting. \n    Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden patterns, structures, or groupings, such as clustering similar customer behaviors or reducing data dimensions. \n    Reinforcement Learning: Agents learn to make sequences of decisions by receiving rewards or penalties for their

In [8]:
### directory Loader
from langchain_community.document_loaders import DirectoryLoader

## load all the text files from the directory
dir_loader = DirectoryLoader(
    "../data/text_files", 
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=False
)

documets = dir_loader.load()
documets


[Document(metadata={'source': '..\\data\\text_files\\python_intro.txt'}, page_content='Machine Learnign basics:\n    Machine learning is a subset of artificial intelligence where algorithms analyze data to learn patterns and make predictions or decisions without\n    being explicitly programmed for every task.  Unlike traditional programming, which relies on fixed rules, machine learning models improve their accuracy over time as they process more experience (data). \n\n    Supervised Learning: Models are trained on labeled data (data with known answers) to predict outcomes for new inputs, commonly used for classification and regression tasks like spam detection or price forecasting. \n    Unsupervised Learning: Algorithms analyze unlabeled data to discover hidden patterns, structures, or groupings, such as clustering similar customer behaviors or reducing data dimensions. \n    Reinforcement Learning: Agents learn to make sequences of decisions by receiving rewards or penalties for th

###Embadings and chunckings

In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Tuple
from sklearn.metrics.pairwise import cosine_similarity



c:\Users\Niraj\Desktop\udemy-agentic-ai\YTRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [20]:
class EmbaddingManager:
    """Handles document embedding generation using SentenceTransformer."""

    def __init__(self, model_name: str = "sentence-transformers/all-MiniLM-L6-v2"):
        """Initialize the EmbaddingManager with a specified model name."""
        self.model_name = model_name
        self.model = None
        self.load_model()

    def load_model(self):
        """Load the SentenceTransformer model."""
        try:
            self.model = SentenceTransformer(self.model_name, token=False)
            print(f"Model '{self.model_name}' loaded successfully.")
        except Exception as e:
            print(f"Error occurred while loading the model: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """Generate embeddings for a list of texts.
        
        Args:
            texts (List[str]): A list of strings for which to generate embeddings.
        
        Returns:
            np.ndarray: An array of embeddings.
        """
        if self.model is None:
            raise ValueError("Model not loaded. Please load the model first.")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Embeddings generated for {embeddings.shape} texts.")
        return embeddings
    

    def get_embading_dimensions(self) -> int:
        """Get the dimensionality of the embeddings generated by the model.
        
        Returns:
            int: The dimensionality of the embeddings.
        """
        if self.model is None:
            raise ValueError("Model not loaded. Please load the model first.")
        
        return self.model.get_sentence_embedding_dimension()
    
 
## intialize embadinganager

embadding_manager = EmbaddingManager()
embadding_manager


Loading weights: 100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 103/103 [00:00<00:00, 6883.25it/s]


Model 'sentence-transformers/all-MiniLM-L6-v2' loaded successfully.


In [29]:
### VectorStore
from langchain_core.documents import Document

class VectorStore:
    """Manage document embeddings and retrieval using ChromaDB."""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """Initialize the VectorStore with a ChromaDB collection."""
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self.initialize_store()

    def initialize_store(self):
        """Initialize the ChromaDB persistent client and collection."""
        try:
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "Collection for storing document embeddings"}
            )
            print(f"ChromaDB collection '{self.collection_name}' initialized successfully.")
            print(f"Persist directory: {self.persist_directory}")
        except Exception as e:
            print(f"Error occurred while initializing ChromaDB: {e}")
            raise

    def add_documents(self, documents: List[Document], embadding_manager: EmbaddingManager):
        """Add documents to the vector store after generating embeddings."""
        if not documents:
            print("No documents to add.")
            return

        texts = [doc.page_content for doc in documents]
        embeddings = embadding_manager.generate_embeddings(texts)

        ids = [str(uuid.uuid4()) for _ in documents]
        metadatas = [doc.metadata or {} for doc in documents]
        embeddings_list = embeddings.tolist() if hasattr(embeddings, "tolist") else embeddings

        self.collection.add(
            ids=ids,
            documents=texts,
            metadatas=metadatas,
            embeddings=embeddings_list,
        )
        print(f"Added {len(documents)} documents to the vector store.")

    def search(self, query: str, embadding_manager: EmbaddingManager, n_results: int = 3):
        """Search the vector store for documents similar to a query."""
        query_embedding = embadding_manager.generate_embeddings([query])
        query_embedding = query_embedding.tolist() if hasattr(query_embedding, "tolist") else query_embedding
        return self.collection.query(
            query_embeddings=query_embedding,
            n_results=n_results,
            include=["documents", "metadatas", "distances"],
        )

    def count(self) -> int:
        """Return the number of documents stored in the collection."""
        return self.collection.count()

